In [29]:
import sklearn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_auc_score


df_local = pd.read_csv("/content/appointments_clean.csv")
#print(df_local.head())

reminder_sent_mapping = {"Yes": 1, "No": 0}
df_local['reminder_sent'] = df_local['reminder_sent'].map(reminder_sent_mapping)
print(df_local)

#print(df_local["visit_type"].value_counts(dropna=False))

#print(df_reminder)
df_encoded = pd.get_dummies( df_local, columns=["booking_channel","visit_type"], drop_first=False, dtype=int)
#print(df_encoded)

X = df_encoded.drop(columns=['patient_id', 'appointment_id', 'clinic_id', 'gender', 'no_show', 'status', 'booked_on', 'appt_date'])
y = df_encoded["no_show"]
print(df_encoded)
print(f"the columns for training X are \n {X}")
print(f"the columns for test Y are \n {y}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"the columns for training X are \n {X_train}")
print(f"the columns for training Y are \n {y_train}")
print(f"the columns for test X are \n {X_test}")
print(f"the columns for test Y are \n {y_test}")
count = (y_test == 1).sum()
#count = y['no_show'].value_counts().get(1, 0)
print(f"\n The number of no shows in the test set are {count}")

#=========Part 3.2===========================================
#============================================================
#============================================================
col_name = 'reminder_sent'
col_index = X.columns.get_loc(col_name)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

#print(f"Std deviation is {scaler.scale_}")
reminder_std = scaler.scale_[col_index]
print(f"The standard deviation for reminder_sent is {reminder_std}")

classifier_model = LogisticRegression(max_iter=10000)
classifier_model.fit(X_train, y_train)
class_predictions = classifier_model.predict(X_test)
class_probs = classifier_model.predict_proba(X_test)[:, 1]

print()
accuracy = accuracy_score(y_test, class_predictions)
print(f"Model Accuracy: {accuracy * 100:.2f}%\n")


cm = confusion_matrix(y_test, class_predictions)
#tn, fp, fn, tp = cm.ravel()
#preds = model.predict(X_test)
#probs = model.predict_proba(X_test)[:, 1]

tn, fp, fn, tp = confusion_matrix(y_test, class_predictions, labels=[0, 1]).ravel()
print("TP:", tp, "TN:", tn, "FP:", fp, "FN:", fn)

precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
print("Precision:", precision, "Recall:", recall, "F1:", f1)

auc = roc_auc_score(y_test, class_probs)
print("ROC-AUC:", auc)

print("Confusion matrix:", cm)
print()


print(f"Coefficients for features are {classifier_model.coef_}")
print()
reminder_coeff = classifier_model.coef_[0][col_index]
print(f"The coefficient for reminder_sent is {reminder_coeff}")
print()

temp_coeff = reminder_coeff/reminder_std
print(f"The true coeff of reminder sent is {temp_coeff}")
print()

non_scaled_coeff = np.exp(temp_coeff)
print(f"The non scaled coefficient is {non_scaled_coeff}")


#==================3.3=============
#==================================
#==================================
from sklearn.tree import DecisionTreeClassifier

train_accuracies = []
test_accuracies = []
train_accuracies_depth = []
test_accuracies_depth = []

X_train_dt, X_test_dt, y_train_dt, y_test_dt = train_test_split(X, y, test_size=0.2, random_state=42)

tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train_dt, y_train_dt)

#Training accuracy
train_pred = tree.predict(X_train_dt)
train_acc = accuracy_score(y_train_dt, train_pred)
train_accuracies.append(train_acc)

# Testing Accuracy
test_pred = tree.predict(X_test_dt)
test_acc = accuracy_score(y_test_dt, test_pred)
test_accuracies.append(test_acc)

print()
print(f"Train Accuracy: {train_acc}")
print(f"Test Accuracy: {test_acc}")

# For model with depth as 3
tree_depth = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_depth.fit(X_train_dt, y_train_dt)

#Training accuracy
depth_train_pred = tree_depth.predict(X_train_dt)
depth_train_acc = accuracy_score(y_train_dt, depth_train_pred)
train_accuracies_depth.append(depth_train_acc)

# Testing Accuracy
depth_test_pred = tree_depth.predict(X_test_dt)
depth_test_acc = accuracy_score(y_test_dt, depth_test_pred)
test_accuracies_depth.append(depth_test_acc)

print(f"Train Accuracy with depth of 3 is: {depth_train_acc}")
print(f"Test Accuracy with depth of 3 is : {depth_test_acc}")



    appointment_id patient_id clinic_id   booked_on   appt_date  age gender  \
0            A0001       P509       C05  2025-01-24  2025-01-25   60      F   
1            A0002       P046       C07  2025-05-27  2025-06-06   39      M   
2            A0003       P009       C02  2025-03-24  2025-03-30   63      F   
3            A0004       P482       C01  2025-04-18  2025-04-22   52      F   
4            A0005       P251       C01  2024-11-23  2025-01-02   64      F   
..             ...        ...       ...         ...         ...  ...    ...   
985          A0996       P415       C05  2025-01-09  2025-01-31   20      M   
986          A0997       P247       C05  2025-02-23  2025-02-27   77      M   
987          A0998       P422       C06  2025-05-05  2025-05-16   58      M   
988          A0999       P509       C05  2025-03-30  2025-04-12   59      F   
989          A1000       P085       C08  2024-12-03  2025-01-17   38      M   

     distance_km  fee_inr booking_channel    visit_